# 01. Exploratory Data Analysis (EDA) - Credit Risk Evaluator

**Autor:** Cientista de Dados & Economista  
**Objetivo de Negócio:** Analisar os determinantes de risco de crédito, comportamento dos tomadores, correlações econômicas e distribuições de variáveis do portfólio para fundamentar a esteira de concessão de crédito.

---

### Tópicos Abordados:
1. Carregamento e Visão Geral dos Dados
2. Análise da Variável Alvo (`loan_status` - Desbalanceamento)
3. Análise Univariada: Variáveis Numéricas e Categóricas
4. Tratamento de Valores Ausentes e Detecção de Outliers
5. Análise Bivariada de Risco (Taxa de Inadimplência por Categoria)
6. Matriz de Correlação e Conclusões da EDA


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

# Configurações de exibição
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: '%.3f' % x)
print("Ambiente configurado com sucesso!")

## 1. Carregamento e Inspeção Inicial do Dataset

O dataset contém dados demográficos e financeiros de proponentes de empréstimos pessoais.

In [ ]:
data_path = '../data/raw/credit_risk_dataset.csv'
if not os.path.exists(data_path):
    data_path = 'data/raw/credit_risk_dataset.csv'

df = pd.read_csv(data_path)
print(f"Dimensões do dataset: {df.shape[0]:,} linhas e {df.shape[1]} colunas.")
df.head()

In [ ]:
# Tipos de dados e contagem de nulos
df_info = pd.DataFrame({
    'Tipo': df.dtypes,
    'Valores Nulos': df.isnull().sum(),
    '% Nulos': (df.isnull().sum() / len(df) * 100).round(2),
    'Valores Únicos': df.nunique()
})
df_info

## 2. Análise da Variável Target (`loan_status`)

A variável alvo representa se o cliente entrou em default (1) ou quitou o empréstimo (0) em um horizonte de 12 meses.  
Em crédito bancário, o desbalanceamento de classes é típico (a maioria dos clientes paga suas dívidas).

In [ ]:
target_counts = df['loan_status'].value_counts()
target_pct = df['loan_status'].value_counts(normalize=True) * 100

print(f"Adimplentes (0): {target_counts[0]:,} ({target_pct[0]:.2f}%)")
print(f"Inadimplentes (1): {target_counts[1]:,} ({target_pct[1]:.2f}%)")
print(f"Taxa Base de Default (Bad Rate): {target_pct[1]:.2f}%")

## 3. Distribuição das Variáveis Numéricas

Analisamos assimetria, valores extremos e estatísticas descritivas (média, mediana, percentis).

In [ ]:
num_cols = ['person_age', 'person_income', 'person_emp_length', 
            'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length']
df[num_cols].describe(percentiles=[0.05, 0.25, 0.50, 0.75, 0.95, 0.99])

## 4. Análise Bivariada de Risco (Taxa de Default por Segmento)

Avaliamos como a probabilidade de inadimplência varia segundo:
1. Moradia (`person_home_ownership`)
2. Finalidade do Empréstimo (`loan_intent`)
3. Histórico de Inadimplência Anterior (`cb_person_default_on_file`)

In [ ]:
# Taxa de default por tipo de moradia
home_risk = df.groupby('person_home_ownership')['loan_status'].agg(
    total='count',
    bads='sum',
    bad_rate='mean'
).reset_index()
home_risk['bad_rate_pct'] = (home_risk['bad_rate'] * 100).round(2)
home_risk.sort_values(by='bad_rate', ascending=False)

In [ ]:
# Taxa de default por finalidade do crédito
intent_risk = df.groupby('loan_intent')['loan_status'].agg(
    total='count',
    bads='sum',
    bad_rate='mean'
).reset_index()
intent_risk['bad_rate_pct'] = (intent_risk['bad_rate'] * 100).round(2)
intent_risk.sort_values(by='bad_rate', ascending=False)

In [ ]:
# Taxa de default por histórico prévio no bureau
bureau_risk = df.groupby('cb_person_default_on_file')['loan_status'].agg(
    total='count',
    bads='sum',
    bad_rate='mean'
).reset_index()
bureau_risk['bad_rate_pct'] = (bureau_risk['bad_rate'] * 100).round(2)
bureau_risk

## 5. Análise de Correlação Linear (Matriz de Spearman / Pearson)

Análise da interdependência entre as variáveis contínuas e o risco de default.

In [ ]:
corr_matrix = df[num_cols + ['loan_status']].corr(method='spearman')
corr_matrix['loan_status'].sort_values(ascending=False)

## 6. Principais Conclusões e Insights para Modelagem

1. **Comprometimento de Renda (`loan_percent_income`):** É a variável com maior correlação positiva com a probabilidade de default. Clientes que comprometem mais de 35% de sua renda apresentam risco desproporcionalmente maior.
2. **Histórico no Bureau (`cb_person_default_on_file`):** Tomadores com histórico prévio de inadimplência apresentam taxa de default significativamente maior.
3. **Tipo de Moradia:** Proponentes que moram de aluguel (`RENT`) apresentam maior volatilidade e taxa de inadimplência superior a tomadores com imóvel quitado (`OWN`) ou hipotecado (`MORTGAGE`).
4. **Valores Ausentes:** `loan_int_rate` (~6% ausentes) e `person_emp_length` (~4% ausentes) demandam imputação por mediana para preservar robustez estatística.
5. **Desbalanceamento:** Com taxa de inadimplência em torno de 25%, abordagens como ponderação de classes (`scale_pos_weight` / `class_weight='balanced'`) serão obrigatórias no treinamento.